In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

In [6]:
df = pd.read_csv('heart_disease_uci.csv')

In [9]:
# 1. Binarizar target
df['target'] = (df['num'] > 0).astype(int)

# 2. Reemplazar ceros en chol
df['chol'] = df['chol'].replace(0, np.nan)

# 3. Reemplazar valores implausibles
df['trestbps'] = df['trestbps'].replace(0.0, np.nan)
df['oldpeak'] = df['oldpeak'].apply(lambda x: np.nan if x < -2 else x)
df['chol'] = df['chol'].apply(lambda x: np.nan if (x < 100 and x > 0) else x)

In [10]:
# Columns excluded from the model:
# - id: row identifier, no clinical meaning
# - dataset: institution of origin, would cause data leakage
# - ca: 66.4% missing, invasive procedure not performed across institutions
# - thal: 52.8% missing, same reason as ca
# - num: original multiclass target, replaced by binary 'target'

features = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 
            'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'target']

df_model = df[features].copy()
print(df_model.shape)
print(df_model.isnull().sum())

(920, 12)
age           0
sex           0
cp            0
trestbps     60
chol        203
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      63
slope       309
target        0
dtype: int64


In [13]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = ['sex', 'cp', 'restecg', 'slope']

encoder = OrdinalEncoder()
df_model[cat_cols] = encoder.fit_transform(df_model[cat_cols])

df_model['fbs'] = df_model['fbs'].astype(float)
df_model['exang'] = df_model['exang'].astype(float)

print(df_model.dtypes)

age           int64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalch      float64
exang       float64
oldpeak     float64
slope       float64
target        int64
dtype: object


In [15]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(max_iter=10, random_state=42)

df_imputed = pd.DataFrame(
    imputer.fit_transform(df_model),
    columns=df_model.columns
)

print(df_imputed.isnull().sum())

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
target      0
dtype: int64


## 6.1 Missing Value Imputation

### Strategy
Missing values were handled using **Iterative Imputation** 
(IterativeImputer, scikit-learn), which predicts each missing 
value using all other variables as predictors via iterative 
regression — a more rigorous approach than simple median 
imputation, which assumes a fixed value regardless of the 
patient's clinical profile.

### Excluded Variables

| Variable | Missing | Reason for Exclusion |
|----------|---------|----------------------|
| `ca` | 611 (66.4%) | >50% missing — invasive fluoroscopy procedure not performed across institutions |
| `thal` | 486 (52.8%) | >50% missing — nuclear stress test not performed across institutions |
| `dataset` | 0 | Excluded to prevent data leakage — institution of origin should not predict disease |
| `id` | 0 | Row identifier, no clinical meaning |
| `num` | 0 | Original multiclass target, replaced by binary `target` |

### Imputed Variables

| Variable | Missing Before | % |
|----------|---------------|---|
| `slope` | 309 | 33.6% |
| `chol` | 203 | 22.1% |
| `fbs` | 90 | 9.8% |
| `trestbps` | 60 | 6.5% |
| `oldpeak` | 63 | 6.9% |
| `thalch` | 55 | 6.0% |
| `exang` | 55 | 6.0% |
| `restecg` | 2 | 0.2% |

After imputation, the dataset contains **920 complete observations** 
with no missing values across all 11 features and 1 target variable.

**Note:** Iterative imputation was applied after encoding categorical 
variables to numeric format using OrdinalEncoder, as the algorithm 
requires numeric input. The original categorical mappings are 
preserved for interpretability.